In [1]:
# Cargar librerias necesarias
import numpy as np
import plotly.graph_objects as go

In [2]:
def loadObject(filepath):
    """Carga un archivo .obj y retorna vertices y caras (indices 0-based)."""
    vertices = []
    faces = []

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()

            if line.startswith('v '): # Vertice
                parts = line.split()
                vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif line.startswith('f '): # Cara
                parts = line.split()[1:]
                # Cada parte puede ser v, v/vt, v//vn, v/vt/vn
                idx = [int(p.split('/')[0]) - 1 for p in parts] # 0-based
                faces.append(idx)
                
    return np.array(vertices), faces

ruta = '../threejs/public/cube.obj'
vertices, faces = loadObject(ruta)
print(f'Archivo cargado: {ruta}')

Archivo cargado: ../threejs/public/cube.obj


In [3]:
# Calcular aristas unicas a partir de las caras
edgeSet = set()

for face in faces:
    n = len(face)
    for i in range(n):
        e = tuple(sorted((face[i], face[(i + 1) % n])))
        edgeSet.add(e)

numVertices = len(vertices)
numEdges    = len(edgeSet)
numFaces    = len(faces)

print('Informacion estructural del modelo')
print('-----------------------------------')
print(f'Vertices : {numVertices}')
print(f'Aristas  : {numEdges}')
print(f'Caras    : {numFaces}')
print(f'Euler (V - A + C) = {numVertices} - {numEdges} + {numFaces} = {numVertices - numEdges + numFaces}')

Informacion estructural del modelo
-----------------------------------
Vertices : 8
Aristas  : 18
Caras    : 12
Euler (V - A + C) = 8 - 18 + 12 = 2


In [4]:
# Caras (triangulos / polígonos)
# Plotly Mesh3d espera listas de indices i, j, k (solo triangulos)
# Si las caras tienen mas de 3 vertices, las triangulamos via fan
triI, triJ, triK = [], [], []
for face in faces:
    for t in range(1, len(face) - 1):
        triI.append(face[0])
        triJ.append(face[t])
        triK.append(face[t + 1])

traceFaces = go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=triI,
    j=triJ,
    k=triK,
    color='steelblue',
    opacity=0.35,
    name='Caras',
    showlegend=True,
    hovertemplate='<b>Cara</b><br>(%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>'
)

# Aristas, Plotly no tiene un tipo arista nativo; se usa Scatter3d con segmentos separados por None
edgeX, edgeY, edgeZ = [], [], []
for (a, b) in edgeSet:
    edgeX += [vertices[a, 0], vertices[b, 0], None]
    edgeY += [vertices[a, 1], vertices[b, 1], None]
    edgeZ += [vertices[a, 2], vertices[b, 2], None]

traceEdges = go.Scatter3d(
    x=edgeX, y=edgeY, z=edgeZ,
    mode='lines',
    line=dict(color='black', width=2),
    name='Aristas',
    hoverinfo='skip' #las aristas no necesitan hover propio
)

# Vertices, Etiqueta de cada vertice para el hover
vertexLabels = [f'V{i}<br>({v[0]:.3f}, {v[1]:.3f}, {v[2]:.3f})' for i, v in enumerate(vertices)]

traceVertices = go.Scatter3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    mode='markers',
    marker=dict(color='crimson', size=6),
    name='Vertices',
    text=vertexLabels,
    hovertemplate='%{text}<extra></extra>'
)

# Figura
fig = go.Figure(data=[traceFaces, traceEdges, traceVertices])

fig.update_layout(
    title=dict(text='Visualizacion interactiva del modelo OBJ', x=0.5),
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data' # Mantiene proporciones reales
    ),
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()